# Lab 5 — Give It Hands
**Session 5 · Tool use + choosing the approach · TCE**

Today: fix Session 1's broken math with a real calculator, chain tools, then decide when tools are even the right answer. **File → Save a copy in Drive.**

In [ ]:
# Cell 1 — setup
%pip install -q -U google-genai
from getpass import getpass
from google import genai
from google.genai import types
import time

client = genai.Client(api_key=getpass("Gemini API key: "))
MODEL = "gemini-flash-latest"  # the free tier's current Flash (July 2026 → Gemini 3.5 Flash). 503 'high demand'? swap to "gemini-flash-lite-latest".
print("ready ✓")

## Part A — The calculator tool

The model never runs your function — it *asks* to, your code executes, the result goes back. The SDK reads your **docstring + type hints** to build the tool declaration, so write them like instructions.

In [ ]:
# Cell 2 — define a tool as a plain Python function
def calculator(expression: str) -> float:
    """Evaluate a mathematical expression and return the exact result.
    Use this for ANY arithmetic. Never compute numbers yourself.
    Example expression: '2347 * 0.18'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:      # validate BEFORE eval — never trust raw args
        raise ValueError(f"unsafe expression: {expression!r}")
    return eval(expression)

# hand the model the tool; SDK does automatic function calling
config = types.GenerateContentConfig(tools=[calculator])

r = client.models.generate_content(
    model=MODEL,
    contents="What is 18% GST on a bill of 2347 rupees, and the total?",
    config=config)
print(r.text)

Ask the same thing **without** `config` (no tool) and compare — watch it guess digits. That contrast is the whole point.

### ✓ Checkpoint 1 — with-tool answer is exact; you saw the no-tool version wobble.

---
## Part B — A second tool + chaining

In [ ]:
# Cell 3 — add a tool; ask something needing BOTH
def days_between(date1: str, date2: str) -> int:
    """Return the number of days between two dates in YYYY-MM-DD format."""
    from datetime import date
    a = date.fromisoformat(date1); b = date.fromisoformat(date2)
    return abs((b - a).days)

config = types.GenerateContentConfig(tools=[calculator, days_between])

r = client.models.generate_content(
    model=MODEL,
    contents=("My internship is from 2026-05-15 to 2026-07-20 and pays 25000 per month. "
              "How many days is it, and roughly how much total if a month is 30 days?"),
    config=config)
print(r.text)

### ✓ Checkpoint 2 — it called BOTH tools to answer.

---
## Part C — See the machinery

Turn OFF automatic calling to watch every function call the model requests.

In [ ]:
# Cell 4 — manual loop: see the raw function calls
config = types.GenerateContentConfig(
    tools=[calculator, days_between],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))

r = client.models.generate_content(
    model=MODEL, contents="What is 15% of 8400 plus 200?", config=config)

for part in r.candidates[0].content.parts:
    if part.function_call:
        print("MODEL WANTS:", part.function_call.name, dict(part.function_call.args))
    elif part.text:
        print("MODEL SAYS:", part.text)
# This is the request the model emits — YOUR code decides whether to run it.

## Part D — Scenario cards (paper + pen, no code)

For each, pick **Prompt / RAG / Tools / Fine-tune** and justify in 2 sentences. This is a mini design doc — and capstone rehearsal.

1. A bot that answers questions about your college's 80-page attendance & exam rulebook, with citations.
2. Replies are correct but too long and too formal; you want short and friendly.
3. "What's the weather in Madurai right now, and should I carry an umbrella to the exam?"
4. A model that must emit your exact 10-field JSON ticket format 50,000×/day on a small cheap model.

### ✓ Checkpoint 3 — defend one choice to me out loud.

---
## Stretch goals

In [ ]:
# Stretch 1 — plug your Session 4 RAG in as a TOOL (capstone move!)
# Paste your search() + chunk_vecs setup from Lab 4 above this cell, then:

def search_notes(query: str) -> str:
    """Search the student's personal course notes and return the most relevant passages.
    Use this for any question about the student's specific courses, syllabus, or college."""
    hits = search(query, k=3)      # from your Lab 4 notebook
    return "\n\n".join(c for s, c in hits)

config = types.GenerateContentConfig(tools=[calculator, search_notes])
r = client.models.generate_content(
    model=MODEL,
    contents="According to my notes, what's the pass mark — and if I have 12/25 internal, what % of the end-sem do I need?",
    config=config)
print(r.text)
# Your capstone is now: knowledge (RAG) + hands (tools). One assistant.

In [ ]:
# Stretch 2 — break it, then fix it
# Give the model a badly-described tool and watch it misfire:
def mystery(x: str) -> str:
    """Does stuff."""        # <- terrible docstring on purpose
    return "42"
# Ask something ambiguous with [calculator, mystery] as tools.
# Then rewrite the docstring to be specific and watch behaviour change.
# Lesson: the docstring IS the prompt.

## Capstone: knowledge + hands + judge

You now have RAG (S4) + tools (S5) + evals (S2). **Save the notebook.** Final session: we attack it, harden it, and you demo. Last break — then the finale.